# 02 - (Optional) Premium prediction practice
The project brief lists a public premium-prediction dataset as **optional practice only**. It has no policy-clause fields, so it is **not** used by the CoverWise risk model.

Place the CSV at `data/raw/acko_dataset_modified.csv` (it is ~219 MB, keep it out of Git - `.gitignore` already excludes it).

In [1]:
import sys
sys.path.insert(0, '..')
from pathlib import Path
import pandas as pd, numpy as np
from coverwise import config
path = config.RAW_DATA_DIR / 'acko_dataset_modified.csv'
assert path.exists(), f'Download the optional dataset to {path}'
df = pd.read_csv(path, nrows=300_000)  # sample for speed
print(df.shape)
df.isna().mean().round(3).sort_values(ascending=False).head(8)

(300000, 20)


Premium Amount          0.345
Previous Claims         0.303
Credit Score            0.116
Number of Dependents    0.092
Customer Feedback       0.065
Health Score            0.061
Annual Income           0.037
Age                     0.015
dtype: float64

About a third of rows have no `Premium Amount` (test rows) - drop them for supervised training.

In [2]:
df = df.dropna(subset=['Premium Amount'])
y = np.log1p(df['Premium Amount'])
X = df.drop(columns=['id','Premium Amount','Policy Start Date'])
cat = X.select_dtypes('object').columns.tolist(); num = X.columns.difference(cat).tolist()
print(len(X), 'rows |', len(cat), 'categorical |', len(num), 'numeric')

196355 rows | 10 categorical | 7 numeric


/tmp/ipykernel_953/964772694.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = X.select_dtypes('object').columns.tolist(); num = X.columns.difference(cat).tolist()


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error
pre = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1), cat)], remainder='passthrough')
model = Pipeline([('pre', pre), ('gb', HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08))])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(Xtr, ytr)
rmsle = mean_squared_error(yte, model.predict(Xte)) ** 0.5
print('RMSLE:', round(rmsle, 4))

RMSLE: 0.935


Premium prediction tends to be weak on this kind of data (premiums are noisy). For CoverWise the important point is: **premium alone does not describe coverage** - which is exactly why the app compares premium change against clause-level changes.